### 09 - Processing dependency trees to html

In this notebook, we will be set up a pipelines to parse dependency visualizations into a html file for manual error checking. 

In [5]:
# FST and CG3 setup
from fst_runtime.fst import Fst
from cg3_process import disambiguate
from dependency_parsing import cg3_to_conllu_batch
import os
os.environ["PATH"] += os.pathsep + "/usr/local/bin"

Set up the paths:

In [6]:


FST_BINARY_PATH = "../data/fst/ojibwe.att"
CG3_GRAMMAR_PATH = "../data/CG3_rules/Ojibwe_updated_dep_parsing.cg3"

OJIBWE_SENTENCES_PATH = "../data/parallel_data/treebank_sentences/ojibwe_gaa.txt"
ENGLISH_SENTENCES_PATH = "../data/parallel_data/treebank_sentences/english_gaa.txt"
TREEBANK_PATH = "../data/treebanks/gaa_sample.conllu"

OUT_HTML_PATH = "../data/dep_parsing_reference/gaa.html"




In [7]:
# change the title of html file through here
HTML_FILE_TITLE = "Gaa Sample July 20"

Building the treebank:

In [8]:
fst = Fst(FST_BINARY_PATH)

# Get sentences in gold standard
sentences_to_parse = []
with open(OJIBWE_SENTENCES_PATH, 'r', encoding="utf-8") as file:
    sentences_to_parse = file.readlines()

cg3_runs: list[str] = []

# Parse sentences and direct output to SYS_TREEBANK_PATH
for sentence in sentences_to_parse:
    disambiguated_sentence = disambiguate(sentence, CG3_GRAMMAR_PATH, fst)
    cg3_runs.append(disambiguated_sentence)
    cg3_to_conllu_batch(disambiguated_sentence, TREEBANK_PATH) 

✓ appended sentence #1 to gaa_sample.conllu
✓ appended sentence #2 to gaa_sample.conllu
✓ appended sentence #3 to gaa_sample.conllu
✓ appended sentence #4 to gaa_sample.conllu
✓ appended sentence #5 to gaa_sample.conllu
✓ appended sentence #6 to gaa_sample.conllu
✓ appended sentence #7 to gaa_sample.conllu
✓ appended sentence #8 to gaa_sample.conllu
✓ appended sentence #9 to gaa_sample.conllu
✓ appended sentence #10 to gaa_sample.conllu
✓ appended sentence #11 to gaa_sample.conllu
✓ appended sentence #12 to gaa_sample.conllu
✓ appended sentence #13 to gaa_sample.conllu
✓ appended sentence #14 to gaa_sample.conllu
✓ appended sentence #15 to gaa_sample.conllu
✓ appended sentence #16 to gaa_sample.conllu
✓ appended sentence #17 to gaa_sample.conllu
✓ appended sentence #18 to gaa_sample.conllu
✓ appended sentence #19 to gaa_sample.conllu
✓ appended sentence #20 to gaa_sample.conllu
✓ appended sentence #21 to gaa_sample.conllu
✓ appended sentence #22 to gaa_sample.conllu
✓ appended sentence

Function to convert to SVG format

In [9]:
# visualise_conllu.py
from pathlib import Path
from typing import Union, List, Tuple
import rich

def assert_tree(doc):
    for tok in doc:
        seen = set()
        cur  = tok
        while cur != cur.head:          # climb to the root
            if cur in seen:
                raise RuntimeError(f"cycle involving «{tok.text}»")
            seen.add(cur)
            cur = cur.head


def sentence_svg(sent, *, compact=True, collapse_punct=True) -> str:
    import spacy
    from spacy.tokens import Doc
    from spacy import displacy

    # build a blank doc
    words  = [tok.form for tok in sent]
    spaces = [True] * (len(words) - 1) + [False]
    nlp    = spacy.blank("xx")
    doc    = Doc(nlp.vocab, words=words, spaces=spaces)

    # 1) copy POS / tag
    for sp_tok, ud_tok in zip(doc, sent):
        sp_tok.pos_ = ud_tok.upos or "X"
        sp_tok.tag_ = ud_tok.xpos or "_"

    # 2) heads & deprels
    for i, (sp_tok, ud_tok) in enumerate(zip(doc, sent)):
        if ud_tok.head == "0":                               # real root
            sp_tok.head = sp_tok
            sp_tok.dep_ = "root"

        elif not ud_tok.deprel or ud_tok.deprel == "dep":    # no arrow for generic dep 
            sp_tok.head = sp_tok
            sp_tok.dep_ = "dep"                              # label kept for info

        else:                                                # normal edge
            head_i = int(ud_tok.head) - 1
            sp_tok.head = doc[head_i]
            sp_tok.dep_ = ud_tok.deprel


    # sanity check
    assert_tree(doc)

    return displacy.render(
        doc,
        style="dep",
        jupyter=False,
        options={"compact": compact, "collapse_punct": collapse_punct},
    )


Build the HTML file

In [14]:
import pyconll, jinja2, rich
from pathlib import Path

def load_lines(path: Path) -> list[str]:
    return [ln.rstrip("\n") for ln in path.read_text(encoding="utf8").splitlines() if ln.strip()]



conllu_path   = Path(TREEBANK_PATH)
oj_path       = Path(OJIBWE_SENTENCES_PATH)
en_path       = Path(ENGLISH_SENTENCES_PATH)
out_html_path = Path(OUT_HTML_PATH)

# 1) read parallel data
trees   = list(pyconll.load_from_file(conllu_path))
ojibwe  = load_lines(oj_path)
english = load_lines(en_path)

#assert len(trees) == len(ojibwe) == len(english), "Parallel files not aligned!"

for s_no, sent in enumerate(trees, 1):
    for t_no, tok in enumerate(sent, 1):
        if not tok.form:
            print(f"Sentence {s_no}, token {t_no} (id={tok.id}) is empty")

rows: list[dict] = []
for i, (tree, oj, en, cg3) in enumerate(zip(trees, ojibwe, english, cg3_runs), 1):
    
    svg = sentence_svg(tree, compact=True, collapse_punct=True)
    rows.append({"no": i, "oj": oj, "en": en, "svg": svg, "cg3": cg3.strip()})

# 2) render HTML via Jinja2 template
tpl = jinja2.Template("""
<!doctype html><html lang="en"><head>
<meta charset="utf-8">
<title>{{ title }} ({{ rows|length }} sentences)</title>
<style>
body{font-family:system-ui,Arial,sans-serif;margin:2rem;}
figure{margin:2rem 0;padding:1rem;border:1px solid #ddd;border-radius:8px;}
figcaption{font-weight:bold;margin-bottom:.5rem;}
.oj{color:#1565c0;} .en{color:#2e7d32;}
.viz svg{width:100%!important;height:auto;}          /* tree */
.cg3{background:#f9f9f9;border:1px solid #eee;
     padding:1rem;margin-top:1rem;
     font:14px/1.4 monospace;white-space:pre-wrap;
     overflow-x:auto;max-height:28em;}               /* scroll if tall */
</style></head><body>
<h1>{{ title }} ({{ rows|length }} sentences)</h1>
{% for r in rows %}
<figure>
 <figcaption>#{{ "%02d"|format(r.no) }}
   <span class="oj">{{ r.oj }}</span><br>
   <span class="en">{{ r.en }}</span>
 </figcaption>

 <div class="viz">{{ r.svg | safe }}</div>
 <pre class="cg3">{{ r.cg3 | e }}</pre>

</figure>
{% endfor %}
</body></html>""")


out_html_path.write_text(tpl.render(rows=rows, title=HTML_FILE_TITLE), encoding="utf8")
rich.print(f"[bold green]✔ Wrote booklet to {out_html_path} ({len(rows)} sentences)")

✔ Wrote booklet to ../data/dep_parsing_reference/gaa.html (150 sentences)